In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from Fonctions.plot_corr_circles import plot_correlation_circle
from Fonctions.plot_outliers import plot_outliers
from Fonctions.plot_outliers import plot_outliers_non_nuls

In [ ]:
df = pd.read_csv('./Downloads/games.csv')



df.columns = ['Name', 'Release date', 'Estimated owners', 'Peak CCU',
       'Required age', 'Price', 'Discount', 'DLC count', 'About the game',
       'Supported languages', 'Full audio languages', 'Reviews',
       'Header image', 'Website', 'Support url', 'Support email', 'Windows',
       'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score',
       'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations',
       'Notes', 'Average playtime forever', 'Average playtime two weeks',
       'Median playtime forever', 'Median playtime two weeks', 'Developers',
       'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']

In [ ]:
inutiles = ["About the game", "Reviews", "Website", "Support url", "Support email", 
            "Metacritic url", "Notes", "Score rank", "Movies", "Header image", "Screenshots", "User score"]
df = df.drop(columns=inutiles)
df.info()

## Conversion des langages en nombre de langages.

In [ ]:
import re

def count_languages(row):
    langs = set()
    for col in ["Supported languages", "Full audio languages"]:
        val = str(row[col])
        # Extraire le contenu entre [ et ]
        matches = re.findall(r'\[([^\]]*)\]', val)
        for match in matches:
            if match.strip():
                # Compter les éléments par les virgules
                items = [x.strip().strip("'") for x in match.split(',')]
                langs.update(items)
    return len(langs)

df["Language Count"] = df.apply(count_languages, axis=1)

## Removing duplicates

We want to remove the **rows** where the Names, Developers and Publishers are the sames because we consider them to be the same game.

We only keep the one with the highest peak CCU.

In [ ]:
df.shape

In [ ]:
df = df.loc[
    df.groupby(['Name', 'Developers', 'Publishers'], dropna=False)['Peak CCU'].idxmax()
]

In [ ]:
df.shape

Nous avons donc supprimé ~150 jeux qui était très probablment des duplicatas.

# Grouping release years

Il n'y a pas suffisamment de jeux avec comme Release Year < 2013 donc nous décidons de les regrouper en une seule classe. 

In [ ]:
years = pd.DatetimeIndex(pd.to_datetime(df["Release date"])).year
df["Year of Release"] = years
df["Year of Release"].value_counts()

In [ ]:
df.loc[df["Year of Release"] <= 2013, "Year of Release"] = 2013
df.loc[df["Year of Release"] == 2026, "Year of Release"] = 2025

In [ ]:
df["Year of Release"].value_counts().sort_index()

In [ ]:
df["Year of Release"] = df["Year of Release"].astype('str')

In [ ]:
df = df.drop(columns="Release date")

# Création de la colonne TagGenre

Ici on cherche à combiner les deux colonnes "Tags" et "Genres" car il y a beaucoup de duplicatas.

Les Tags et Genres étant stockés dans un string séparée par des virgules, nous les séparons en différentes colonnes, c'est une forme de one hot encoding.

In [ ]:
def merge_and_clean(row):
    tags = str(row['Tags']).split(',') if pd.notna(row['Tags']) else[]
    genres = str(row['Genres']).split(',') if pd.notna(row['Genres']) else[] 
    
    tous_les_mots =[mot.strip() for mot in (tags + genres) if mot.strip()]
    mots_uniques = set(tous_les_mots)
    return ','.join(mots_uniques)
    
df2 = df.copy(deep = True)
combined_series = df2.apply(merge_and_clean, axis=1)
combined_dummies = combined_series.str.get_dummies(sep=',').astype(bool)
combined_dummies.columns =[f"TagGenre_{c}" for c in combined_dummies.columns]
df2 = pd.concat([df2, combined_dummies], axis=1)
print(df2.head())

On sépare les colonnes Developers et Publishers car elles possèdent trop de valeurs uniques et donc provoqueraient une explosion du nombre de colonnes lors de leur one hot encoding pour les méthodes de MCA et MFA.

In [ ]:
Optionels = ["Developers", "Publishers"]
df2 = df2.drop(columns=Optionels)
df2.info(verbose=all)

In [ ]:
Categories_dummies = df2['Categories'].str.get_dummies(sep=',').astype(bool)
Categories_dummies.columns = [f"Categories_{c.strip()}" for c in Categories_dummies.columns] # merci gemini
df2 = pd.concat([df2, Categories_dummies], axis=1)
print(df2.head())

# Outliers

In [ ]:
NUM_COLS = df2.select_dtypes(include=["number", "int64"])
outlier_summary = plot_outliers(df2, NUM_COLS.columns)

On observe une grande proportion d'outliers sur certaines variables pour différentes raisons :

La majorité des variables ont en fait un trop grand nombre de 0, ce qui force la borne basse et haute de la détection d'outlier à la même valeur : 0.
Cela provoque la détection de toute valeur non nulle comme étant un outlier.

Il est possible de changer le critère d'outlier pour être par exemple : Le 1% des valeurs les plus extrêmes sont des outliers.

Nous pouvons aussi réaliser la même étude uniquement sur les valeurs non nulles.

In [ ]:
NUM_COLS = df2.select_dtypes(include=["number", "int64"])
outlier_summary = plot_outliers_non_nuls(df2, NUM_COLS.columns)

On décide pour la suite que nous appliquerons des transformations log sur Positive, Negative, Recommandation, et les temps de jeux forever (car les valeurs maximales sont très grandes). 

D'autres transformations seront discutées lors de l'ACP (Peak CCU, Price, Language count et DLC count).

# 5 biggest outliers of quantitative variables

In [ ]:
cols_of_interest = [
    "Median playtime forever",
    "Average playtime forever", 
    "Recommendations",
    "Peak CCU",
    "Positive",
    "Negative",
    "DLC count"
]

def get_top_outliers(df, cols, top_n=5):
    for col in cols:
        serie = df[col]
        Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
        IQR = Q3 - Q1
        mask_outliers = (serie < Q1 - 1.5 * IQR) | (serie > Q3 + 1.5 * IQR)
        
        # Récupérer les outliers avec leur nom
        outliers = df[mask_outliers][["Name", col]].copy()
        outliers = outliers.sort_values(col, ascending=False).head(top_n)
        
        print(f"\n{'='*50}")
        print(f"Top {top_n} outliers pour {col}")
        print(f"{'='*50}")
        print(outliers.to_string(index=False))

get_top_outliers(df2, cols_of_interest, top_n=5)

## Les jeux les plus "détestés"

In [ ]:
cols_of_interest = [
    "Positive",
    "Negative",
]
df_temp = df2[["Name","Positive","Negative"]].copy()
df_temp["Negative - Positive"] = (df2["Negative"] - df2["Positive"])
outliers = df_temp.sort_values("Negative - Positive", ascending=False).head(5)
print(f"{'='*50}")
print(outliers.to_string(index=False))

Après quelques recherches, ces jeux sont pour la plupart cibles de "review bombing" pour une raison particulière. Par exemple pour "Kerbal Space Program 2" il s'agit de l'annonce de la fermeture du studio pour raison financière qui a provoqué ces avis. Pour "Mirror 2: Project X" il s'agit d'une protestation des joueurs sur un choix du studio de ne pas intégrer de contenu pour adultes dans le jeu alors que le prequel en contenait.

# Log scallers on some columns

In [ ]:
df3 = df2.copy(deep = True)
df3["Positive"] = np.log2(df2["Positive"] + 1)
df3["Negative"] = np.log2(df2["Negative"] + 1)
df3["Recommendations"] = np.log2(df2["Recommendations"] + 1)
df3["Average playtime forever"] = np.log2(df2["Average playtime forever"] + 1)
df3["Median playtime forever"] = np.log2(df2["Median playtime forever"] + 1)

# ACP

## Dataset sans les autres log transform

In [ ]:
data_pca_2 = df3.select_dtypes(include=["number", "int64"]) #
scaler = StandardScaler()
scaled_df3 = scaler.fit_transform(data_pca_2)
pca_2 = PCA()
pca_df3 = pca_2.fit_transform(scaled_df3) 
explained_variance_2 = pca_2.explained_variance_ratio_

#feature_names = data_pca_2.columns.tolist()
#plot_correlation_circle(pca, feature_names, dim_pairs=[(0,1), (0,2), (1,2)])

corr_matrix = data_pca_2.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

## Dataset avec les autres log transform sur Peak CCU et DLC count

In [ ]:
df3_withOtherLogs = df3.copy(deep = True)
df3_withOtherLogs["Peak CCU"] = np.log2(df3["Peak CCU"] + 1)
df3_withOtherLogs["DLC count"] = np.log2(df3["DLC count"] + 1)
df3_withOtherLogs["Price"] = np.log2(df3["Price"] + 1)
df3_withOtherLogs["Language Count"] = np.log2(df3["Language Count"] + 1)

In [ ]:
data_pca = df3_withOtherLogs.select_dtypes(include=["number", "int64"])
scaler = StandardScaler()
scaled_df3 = scaler.fit_transform(data_pca)
pca = PCA()
pca_df3 = pca.fit_transform(scaled_df3) 
explained_variance = pca.explained_variance_ratio_

feature_names = data_pca.columns.tolist()

In [ ]:
corr_matrix = data_pca.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

Il est intéressant de noter que les corrélations sont très différentes entre les datasets avec/sans log supplémentaires sur **Price**, **Language Count**, **Peak CCU** et **DLC count**.

Ça laisse donc sous-entendre que les liens entre les variables sont probablement assez complexes et pas simplement linéaires ou bien que des valeurs extrêmes bruitent l'ACP sans log.

**Peak CCU**
Le pic de joueurs simultanés est fortement corrélé aux avis positifs/négatifs (0.63), aux recommandations (0.66), au playtime forever (0.60), et dans une moindre mesure au Metacritic score (0.36) et au nombre de DLC (0.37). Un jeu populaire génère donc globalement plus d'engagement sous toutes ses formes.

**Recommendations et temps de jeu**
Le groupe **Recommendations, Average playtime forever, Median playtime forever** forme un cluster très cohérent (corrélations entre 0.74 et 0.99). Les joueurs qui recommandent un jeu y passent aussi beaucoup de temps ce qui est logique : on recommande ce qu'on a vraiment joué.

**Avis positifs et négatifs symétriques**
**Positive** et **Negative** sont corrélés à 0.91. Un jeu très joué accumule mécaniquement beaucoup d'avis dans les deux sens. Ce sont davantage des indicateurs de volume que de qualité.

**Playtime Two weeks**
Les variables **Average** et **Median playtime two weeks** sont très fortement corrélées entre elles (0.97), tout comme leurs équivalents **forever** (0.99). Mais le playtime **two weeks** et le playtime **forever** sont faiblement liés, ce qui suggère que le temps de jeu récent ne reflète pas nécessairement le temps de jeu cumulé sur le long terme. Un jeu ancien très joué peut avoir peu d'activité récente, et inversement.

**Prix et Language Count peu corrélés**
Le prix et le nombre de langages, bien que corrélés à Recommendations, Peak CCU et les Playtime Forever, les corrélations sont très faibles < 0.25.

**Achivements isolé**
Toutes les corrélations de **Achivements** sont proches de 0, on considère donc qu'il s'agit d'une information complètement différente des autres variables.

**Conclusion générale**

Les variables se regroupent en plusieurs blocs :
- **Popularité** : Peak CCU, Positive, Negative, Recommendations, Average/Median playtime foreverUser score
- **Engagement court terme** : Average/Median playtime two weeks, (corrélations > 0.97 entre eux)
- **Prix** : Price et Language Count sont faiblement corrélées entre elles et aussi avec le bloc de 'Popularité'. 
- **Variables indépendantes** : Achievements est presque complètement independant des autres variables.

Le Metacritic score et le nombre de DLC semblent jouer un rôle modéré sur la popularité.

## Scree Plot

In [ ]:
plt.plot(pca.explained_variance_ratio_.cumsum() * 100)
plt.grid()
plt.show()

On observe une réduction de dimension importante : pour conserver 85 % de la variance, il suffit de conserver 8 dimensions au lieu de 14.

In [ ]:
plot_correlation_circle(pca, feature_names, dim_pairs=[(0,1), (0,2), (1,2)])

La PC1 est assez compliquée à lire mais semble encoder l'engagement d'un jeu, avec notamment le **Peak CCU**, les **Recommendations**, les **Playtime forever**. 

La PC2 encode clairement les variables **Playtime two weeks**.

La PC3 semble, elle, indiquer le prix du jeu avec **Price** et **Discount** (Discount n'est pas réellement encodé dans la PC3 mais la contre-corrélation avec **Price** la fait ressortir du lot). Cependant la PC3 n'explique déjà plus que seulement 7 % de la variance des données. Il semble que **Achivements** soit aussi légèrement encodé dans la PC3 mais très peu (comme **Discount**)

### Individuals Factor Map 

In [ ]:
## selecting the first 9 principal components

i = 9
pca_games = pca_df3[:,0:i-1]

expl_var = np.sum(pca.explained_variance_ratio_[:i-1])
print(f"The first {i} PC represent {expl_var*100:.1f}% of the variance")

In [ ]:
import matplotlib

fig, axes = plt.subplots(2,3,figsize = (12,8))

pairs = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]

for idx, (i,j) in enumerate(pairs):

        ax = axes.flatten()[idx]
        ax.scatter(pca_games[:,i],  pca_games[:,j], s = 1, alpha = 0.4, color = "steelblue")

        ax.set_title("Individuals factor map — PCA")
        ax.set_xlabel(f"PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}%)")
        ax.set_ylabel(f"PC{j+1} ({pca.explained_variance_ratio_[j]*100:.1f}%)")
        ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
        ax.axvline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

# Clustering on Individuals

In [ ]:
from sklearn.preprocessing import StandardScaler

df_all = df2.copy(deep = True)

df_cluster_copy = df3_withOtherLogs 

raw_games = df3_withOtherLogs.select_dtypes(include=["int64", "number"]) # raw data for k means on raw data

pca_games_scaled = StandardScaler().fit_transform(pca_games) # scaling data for kmeans on scaled data

datasets = [pca_games, pca_games_scaled, raw_games]


### K-means on numerical data

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from matplotlib import colors

from yellowbrick.cluster import SilhouetteVisualizer

#Using the Silhouette Score two find the optimal k number of clusters 

fig, ax = plt.subplots(5, 2, figsize=(15,8))

for k in range(2, 12):
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init='auto', max_iter=100, random_state=42)
    q, mod = divmod(k, 2)
    
    # Create SilhouetteVisualizer instance with KMeans instance
    visualizer = SilhouetteVisualizer(kmeans, colors='yellowbrick', ax=ax[q-1][mod], force_model=True)
    visualizer.fit(pca_games)

In [ ]:
### k means over pca data

pca_scores = [0,0]
scaled_scores = [0,0]
raw_scores = [0,0]

for k in range(2, 30):
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init='auto', max_iter=100, random_state=42)
    
    labels_pca = kmeans.fit_predict(pca_games)
    labels_scaled = kmeans.fit_predict(pca_games_scaled)
    labels_raw = kmeans.fit_predict(raw_games)
    
    score_pca = silhouette_score(pca_games, labels_pca)
    score_scaled = silhouette_score(pca_games_scaled, labels_scaled)
    score_raw = silhouette_score(raw_games, labels_raw)

    pca_scores.append(score_pca)
    scaled_scores.append(score_scaled)
    raw_scores.append(score_raw)


So we will be looking at Kmeans clustering results for K ∈ {2, 11, 20}

In [ ]:
cmap = plt.get_cmap("tab20",3)

x = np.arange(30)

sil_scores = np.zeros((3,30))
sil_scores[0,:] = pca_scores
sil_scores[1,:] = scaled_scores
sil_scores[2,:] = raw_scores

descr = ["PCA", "PCA and Scaled", "Raw Data"]

fig, ax = plt.subplots(figsize = (10,5))

for i in range(3):
    ax.plot(x,sil_scores[i,:],color = cmap.colors[i], linewidth=2, label = descr[i])

ax.set_xlabel("Number of Clusters in KMeans")
ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score for KMeans on Raw/PCA/Scaled PCA data for different K")
ax.set_xlim(0,30)
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
cl_sizes = [2,11,20]

datasets = [pca_games, pca_games_scaled, raw_games]

dataset_names = ["PCA Data", "PCA Data - Unit Variance", "Raw Data"]

cluster_partitions = np.zeros((df.shape[0],9))

i = 0

for data in datasets: 

    for K in cl_sizes: 

        kmeans_games = KMeans(n_clusters=K, init="k-means++", n_init="auto", random_state = 42)
        cluster_partitions[:,i] = kmeans_games.fit_predict(data)

        i+=1

In [ ]:
### Plotting the 6 PCA Partitions for three different Ks and the 2 different data types 

import math

pca_cluster_partitions = cluster_partitions[:,0:6].T
pca_datasets = [x for x in datasets[0:2] for _ in range(3)]

fig, axes = plt.subplots(12,3,figsize = (20,60))

pairs = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]

for index, (data,partition) in enumerate(zip(pca_datasets,pca_cluster_partitions)):

        K = cl_sizes[index%3]

        cmap = plt.get_cmap('tab20',K)

        for idx, (i,j) in enumerate(pairs):
                
                ax = axes.flatten()[6 * index +idx]
                sc = ax.scatter(data[:,i],  data[:,j], c = partition, s = 1, alpha = 0.6, cmap = cmap)

                ax.set_title(f"Individuals factor map of {dataset_names[math.floor(index/3)]} - colored by KMeans Clustering with K={K}")
                ax.set_xlabel(f"PC{i+1}")
                ax.set_ylabel(f"PC{j+1}")
                ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
                ax.axvline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
                ax.spines[["top", "right"]].set_visible(False)
                ax.grid(True, linestyle="--", alpha=0.3)

                fig.colorbar(sc, ax=ax, ticks=range(K), label= "Games Cluster")

plt.tight_layout()
plt.show()

In [ ]:
### Clusters in a UMAP Projection for PCA Data

### to be executed and calculated during night!


import umap.umap_ as umap 

reducer = umap.UMAP(n_neighbors= 15, min_dist=0.1, random_state=42)

games_pca_umap = reducer.fit_transform(pca_games)
games_scaled_umap = reducer.fit_transform(pca_games_scaled)
games_raw_umap = reducer.fit_transform(raw_games)

umaps = [games_pca_umap, games_scaled_umap, games_raw_umap]

In [ ]:
## Plot the Clusters in the UMAP Projection 

fig, axes = plt.subplots(5,2, figsize=(24,48))

for idx, umap in enumerate(umaps):

    for i in range(3):
        
        index = idx*3 +i
        K = cl_sizes[i]

        cmap = plt.get_cmap('tab20',K)

        ax = axes.flatten()[index]
        sc = ax.scatter(umap[:,0], umap[:,1], c = cluster_partitions[:,index], s = 5, alpha = 0.6, cmap = cmap)

        ax.set_title(f"UMAP Projection of {dataset_names[idx]} colored by K-Means Clustering with K = {K}")
        ax.set_xlabel("UMAP Dim1")
        ax.set_ylabel("UMAP Dim2")

        fig.colorbar(sc, ax=ax, ticks=range(K), label= "Games Cluster")

fig.delaxes(axes.flatten()[9])
plt.tight_layout()
plt.show()

In [ ]:
# Number of Games in each cluster depending on 4 different K's

fig, axes = plt.subplots(5,2,figsize = (14,20))

for i in range(9):
    cmap = plt.get_cmap('tab20', cl_sizes[i%3])
    ax = axes.flatten()[i]
    ax.bar(*np.unique(cluster_partitions[:,i], return_counts=True), color=cmap.colors)
    ax.set_ylabel("Number of Games per cluster")
    ax.set_xlabel("Cluster")
    ax.set_title(f"Number of Clusters: {cl_sizes[i%3]}")

fig.delaxes(axes.flatten()[9])
plt.tight_layout()
plt.show()



In [ ]:
print(df_all.columns.tolist())
df_all.info(verbose = True)

In [ ]:
values = df_all["Estimated owners"].mode().head(3)

results = []
for rank, value in enumerate(values): 

    results.append({
            "cluster": 1,
            "variable": "Estimated owners",
            "rank": rank,
            "value_type": "mode",
            "value": value
    })

print(results)

### Creation of the column "owners_num" to sort the "Estimated Owners" values as it is of dtype object

In [ ]:
def parse_owners(x): 
    low, high = x.replace(",", "").split(" - ")
    return (int(low)+ int(high)) / 2

df_all["owners_num"] = df_all["Estimated owners"].apply(parse_owners)

In [ ]:
def partition_analysis(data, cluster_partition):

    results = []

    variables = ["Estimated owners","Peak CCU", "Required age","Price","Year of Release", "Average playtime two weeks",
                 "Median playtime forever","Language Count","Positive","Negative","Discount", "Recommendations"]

    bool_variables = [(9,12),           ## index 9 is windows index 11 is mac
                      (27,479),         ## thes are the indices for the tags
                      (479,537)]        ## thes are the indices for the categories                 ## pairs of index bounderies for the boolean variables 
    

    for cluster in np.unique(cluster_partition): 

        df_temp = data[cluster_partition == cluster]

        for var in variables: 
            
            if var in ["Year of Release","Required age", "Estimated owners"]:
                values = df_temp[var].mode().head(3)

                for rank, value in enumerate(values):
                    
                    results.append({
                        "cluster": cluster,
                        "variable": var,
                        "rank": rank,
                        "value_type": "mode",
                        "value": value
                })
            
            else: 
                value = df_temp[var].mean()

                results.append({
                        "cluster": cluster,
                        "variable": var,
                        "rank": 1,
                        "value_type": "mean",
                        "value": np.round(value)
                })


        for (i,j) in bool_variables: 

            summed = df_temp.iloc[:,i:j].sum()

            values = summed.sort_values(ascending=False).head(3)
                      
            for rank, (var, value) in enumerate(values.items(), start=1):
                    
                    results.append({
                        "cluster": cluster,
                        "variable": var,
                        "rank": rank,
                        "value_type": "count",
                        "value": value
            })
        
        top_games = df_temp.sort_values("owners_num", ascending=False).head(3)["Name"] 

        for rank, value in enumerate(top_games.values, start=1):
                    
                    results.append({
                        "cluster": cluster,
                        "variable": "top_games",
                        "rank": rank,
                        "value_type": "name",
                        "value": value
            })


    summary_clustering_df = pd.DataFrame(results)

    return summary_clustering_df


In [ ]:
def get_value(df, var):

    vals = df[df["variable"]==var]["value"]
    return vals.iloc[0] if len(vals) > 0 else None

def get_OS(df):
    systems = ["Windows", "Mac", "Linux"]
    
    df_os = df[df["variable"].isin(systems)]
    
    top = df_os[df_os["rank"] == 1]  
    
    if len(top) > 0:
        return top["variable"].iloc[0]
    else:
        return None
    
def get_Genres(df):

    vars = df["variable"].unique()
    genres = [s[9:] for s in vars if "TagGenre" in s]
    return genres 

def get_Categories(df):

    vars = df["variable"].unique()
    categories = [s[11:] for s in vars if "Categorie" in s]

    return categories

def get_Games(df): 
    games = df[df["variable"] =="top_games"]["value"].tolist()

    return games 

In [ ]:
def collect_cluster_results(df):

    results = []

    for cluster in np.unique(df["cluster"]):

        df_temp = df[df["cluster"] == cluster]

        cluster_dict = {
            "cluster": int(cluster),
            "estimated_owners": get_value(df_temp, "Estimated owners"), 
            "peak_ccu": get_value(df_temp, "Peak CCU"),
            "average_playtime_two_weeks": get_value(df_temp, "Average playtime two weeks"),
            "median_playtime_forever": get_value(df_temp, "Median playtime forever"),
            "language_count": get_value(df_temp, "Language Count"),
            "recommendations": get_value(df_temp, "Recommendations"),
            "positives": get_value(df_temp, "Positive"),
            "negatives": get_value(df_temp, "Negative"),
            "required_age": get_value(df_temp, "Required age"),
            "discount": get_value(df_temp, "Discount"),
            "price": get_value(df_temp, "Price"),
            "year": get_value(df_temp, "Year of Release"),
            "os": get_OS(df_temp),
            "genres": get_Genres(df_temp),
            "categories": get_Categories(df_temp),
            "top_games": get_Games(df_temp)
        }

        results.append(cluster_dict)

    return results


In [ ]:
first_frame = partition_analysis(df_all, cluster_partitions[:,5])

first_frame_results = collect_cluster_results(first_frame)

print(first_frame_results)

In [ ]:
def make_labels(results): 

    var_labels = {}

    variables = list(results[0].keys())[1:]

    for var in variables: 

        labels_values = {}

        for dictionary in results:

            labels_values.update({f"{dictionary["cluster"]}": dictionary[var]})

        var_labels.update({var: labels_values})

    return var_labels



In [ ]:
def plot_one_Variable_on_umap(variable_name, labels,umap,dataset_name, cluster_partition, on_cluster):

    K = len(np.unique(cluster_partition))

    fig, ax = plt.subplots(figsize=(14,12))

    cmap = plt.get_cmap('tab20',K)

    if on_cluster:

        sc = ax.scatter(umap[:,0], umap[:,1], c = cluster_partition, s = 5, alpha = 0.6, cmap = cmap)

    for cluster in (np.unique(cluster_partition)):

        mask = cluster_partition == cluster

        cluster_color = cmap(int(cluster))

        label = labels.get(f'{int(cluster)}')

        if not on_cluster:
            
            ax.scatter(umap[mask,0], umap[mask,1], color = cluster_color, s = 5, alpha = 0.6, label = label)

        if on_cluster:

            x = np.median(umap[mask,0]) *1.3
            y = np.median(umap[mask,1]) *1.3

            ax.text(x,y, str(label), fontsize=12, ha="center", va="center", zorder =100, clip_on=False,
                    bbox=dict(boxstyle="round, pad=0.3", facecolor="white", edgecolor=cluster_color, linewidth= 3, alpha = 1))
            
    if not on_cluster: 
        ax.legend(
            bbox_to_anchor=(0.02, 0.98),
            loc='upper left',
            borderaxespad=0.0,
            markerscale = 6,
            framealpha = 1,
            frameon=True,
            facecolor = "white", 
            
    )

    if on_cluster:
        cbar = fig.colorbar(sc, ax=ax, ticks=range(K))
        cbar.set_label("Cluster of Games")


    ax.set_title(f"UMAP Projection of {dataset_name} colored by K-Means, K = {K} and {variable_name} per cluster")
    ax.set_xlabel("UMAP Dim1")
    ax.set_ylabel("UMAP Dim2")

    plt.rcParams['font.family'] = 'DejaVu Sans'
    plt.tight_layout()
    plt.show()

In [ ]:
first_frame = partition_analysis(df_all, cluster_partitions[:,1])

first_frame_results = collect_cluster_results(first_frame)

first_frame_labels = make_labels(first_frame_results)

variabs_to_investigate = list(first_frame_labels.keys())

for var in variabs_to_investigate:

    if var in ["categories", "genres", "top_games"]:
        plot_one_Variable_on_umap(var,first_frame_labels.get(var),umaps[0],dataset_names[0], cluster_partitions[:,1], False)
    else:
        plot_one_Variable_on_umap(var,first_frame_labels.get(var),umaps[0],dataset_names[0], cluster_partitions[:,1], True)

In [ ]:
variabs_to_investigate = list(first_frame_labels.keys())
print(variabs_to_investigate)

In [ ]:
first_frame_labels = make_labels(first_frame_results)

labels = first_frame_labels.get('price')
print(labels)
print(labels.get('1'))

In [ ]:
first_frame_labels.keys()

In [ ]:
first_frame.head(30)



In [ ]:
cluster_analysis = []



for partition in cluster_partitions:

    frame = partition_analysis(df_all, partition)
    cluster_analysis.append(frame)

In [ ]:
### auch die colorbar rechts färben oder eher mit legende arbeiten als den wert reinzuschreiben


In [ ]:
## Clustergrafen für spiele mit gleichen Tags/Genres 


# Prédire via LDA

Nous souhaitons utiliser des variables qualitatives et quantitatives pour prédire une variable qualitative, la LDA est la seule méthode de classification supervisée étudiée cette année qui permet de réaliser cette tache.

La méthode étant supervisée, il est nécessaire de séparer en train et test le dataset et de choisir une variable à prédire. Ici nous cherchons à prédire l'année de publication du jeu, le nombre de **Estimated Owner** et si metacritic à attribuer une note au jeu (ou pas).

In [ ]:
import prince
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
def LDA_train_test(data_LDA, target = "") :
    X = data_LDA.drop(columns=target)
    Y = data_LDA[target]
    
    train_x, test_x, train_y, test_y = train_test_split(X, Y, test_size=0.3, random_state=20)
    
    clf = LDA()
    clf.fit(train_x, train_y)
    
    # --- #
    
    train_pred = clf.predict(train_x)
    test_pred  = clf.predict(test_x)
    
    train_accuracy = accuracy_score(train_y, train_pred)
    test_accuracy  = accuracy_score(test_y, test_pred)
    
    print("--- Linear Discriminant Analysis ---\n")
    print(f"Train accuracy: {train_accuracy:.2%}")
    print(f"Test accuracy:  {test_accuracy:.2%}")
    print(' ')
    print(classification_report(test_y, test_pred))
    # --- #

## Prédiction de Year of release 

In [ ]:
data_LDA = df3.select_dtypes(include=["number", "int64", "bool", "object", "str"])
data_LDA = data_LDA.drop(columns=["Name","Genres","Tags","Categories","Full audio languages","Supported languages"])

le = LabelEncoder()
data_LDA["Estimated owners"] = le.fit_transform(data_LDA["Estimated owners"].astype(str))
le_year = LabelEncoder()
data_LDA["Year of Release"] = le_year.fit_transform(data_LDA["Year of Release"].astype(str))

print("Mapping des classes :", dict(enumerate(le_year.classes_)))

LDA_train_test(target="Year of Release", data_LDA=data_LDA)

Nous observons une classification assez mauvaise (environ 35% de taux de bonne prédiction) mais cela est largement supérieur à un choix aléatoire qui donnerais 7.7% de précision. On observe aussi que l'année 2025+ est bien mieux prédite que les autres à 50% de taux de bonne détection, cela est probablement lier au fait que cette classe est la plus présente dans le dataset.

## Prediction de Estimated Owners

In [ ]:
data_LDA_2 = df3.select_dtypes(include=["number", "int64", "bool", "object", "str"])
data_LDA_2 = data_LDA_2.drop(columns=["Name","Genres","Tags","Categories","Full audio languages","Supported languages"])

le = LabelEncoder()
data_LDA_2["Estimated owners"] = le.fit_transform(data_LDA_2["Estimated owners"].astype(str))
le_year = LabelEncoder()
data_LDA_2["Year of Release"] = le_year.fit_transform(data_LDA_2["Year of Release"].astype(str))

print("Mapping des classes :", dict(enumerate(le.classes_)))

LDA_train_test(target="Estimated owners", data_LDA=data_LDA_2)

Les résultats de la classification sont extrèmement hétérogène, avec les classes (0-0 et 0-20000) proche de 90% de taux de bonne prédiction contre 20% pour les classes plus rares.

Cela confirme l'observation précédente que les classes avec plus d'individus sont mieux prédites que celles avec peu d'individus. 

On trouve tous de même un très bon taux global de prédiction à 75% grandement influencer par le fait que le dataset est composée disproportionnellement par des jeux avec peu de estimated owners (qui sont donc les classes les mieux prédites). 

## Prediction de "metacritic has given a score to the game"

In [ ]:
data_LDA_3 = df3.select_dtypes(include=["number", "int64", "bool", "object", "str"])
data_LDA_3 = data_LDA_3.drop(columns=["Name","Genres","Tags","Categories","Full audio languages","Supported languages"])
data_LDA_3["Metacritic score"] = data_LDA_3["Metacritic score"] != 0

le = LabelEncoder()
data_LDA_3["Estimated owners"] = le.fit_transform(data_LDA_3["Estimated owners"].astype(str))
le_year = LabelEncoder()
data_LDA_3["Year of Release"] = le_year.fit_transform(data_LDA_3["Year of Release"].astype(str))

LDA_train_test(target="Metacritic score", data_LDA=data_LDA_3)

La prédiction ici est très interréssante. Les jeux n'ayant pas de **score metacritic** sont correctement identifiés en énorme majoritée 99% de précision. Mais les jeux ayant reçu un **score metacritic** sont eux prédit à seulement 50% de précision. Le modèle ne prédit donc pas toujour la même valeur mais nous retrouvons encore le même résultats, les classes majoritaires (ici False) sont bien mieux prédites que les autres classe. 